### Load libraries

In [ ]:

from mstr_robotics._paths import REPO_ROOT, CONFIG_DIR, USER_CONFIG, OSI_FILES, OSI_SCHEMA, OSI_DASHBOARD_CONTEXT, MCP_DATA, PYTHON_IO
#from flashtext import KeywordProcessor
from mstr_robotics.navigation import AnswerPrompts, MstrObjects
from mstr_robotics.report import Rep, Prompts, Cube
from mstr_robotics.mstr_classes import get_conn
from IPython.display import HTML
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path
import os
import json 

from mstr_robotics.user_rag import KeywordProcessor, Perplexity

i_prompts=Prompts()
i_rep=Rep()
i_cube=Cube()
u_perplexity=Perplexity()
i_mstr_objects=MstrObjects()

u_keyword_processor=KeywordProcessor()
os_mcp_folder_str=str(MCP_DATA)


user_path=USER_CONFIG
env_file=CONFIG_DIR / "API_KEY.env"

try:
    with open(USER_CONFIG, 'r') as openfile:
        user_d = json.load(openfile)
        conn_params =  user_d["conn_params"]
except Exception as err:
    print(err)
    
try:
    with open(CONFIG_DIR / "jupyter_objects_d.json", "r") as openfile:
        jupyter_objects_d = json.load(openfile)
    nb_d = jupyter_objects_d["jup_Colab_perplex"]
except Exception as err:
    print(err)

try:
    load_dotenv(env_file)
except Exception as err:
    print(err)

### Config

In [2]:
rag_d = jupyter_objects_d["turtorial_RAG"]

msg_t="Please show me the Cost_1, Revenue and Profit for the attributes Year,Category, Region"
msg_t= msg_t + " and filter for the yaer 2021, 2022 and 2023"
msg_t= msg_t + " and Categories starting with B"
msg_t= msg_t + " and Revenue is between 10 and 1000000"

#Name of the generated Report in MSTR
ai_rep_name="dyn_prompt_page_botstat"
ai_rep_folder_id=nb_d["folders"]["ai_rep_folder_id"]
report_id=nb_d["reports"]["report_id"]

# MSTR specific objects and definitions
# Loaded directly from the RAG cubes generated by load_rag_cubes.ipynb
# (previously read from CSV files in the MCP_DATA folder)

project_id = conn_params["project_id"]
cube_attribute_form_elements_id = rag_d["cube_attribute_form_elements_id"]
cube_attribute_elements_id      = rag_d["cube_attribute_elements_id"]
cube_att_form_def_id            = rag_d["cube_att_form_def_id"]
cube_obj_prp_rel_id             = rag_d["cube_obj_prp_rel_id"]

In [3]:
### load RAG cubes

conn = get_conn(**conn_params)

attribute_form_elements_df = i_cube.load_cube_to_df(conn=conn, cube_id=cube_attribute_form_elements_id)
attribute_elements_df      = i_cube.load_cube_to_df(conn=conn, cube_id=cube_attribute_elements_id)
att_form_def_df            = i_cube.load_cube_to_df(conn=conn, cube_id=cube_att_form_def_id)
obj_prp_rel_df             = i_cube.load_cube_to_df(conn=conn, cube_id=cube_obj_prp_rel_id)

i_answer_prompts=AnswerPrompts(attribute_form_elements_df=attribute_form_elements_df
                                , attribute_elements_df=attribute_elements_df
                                , obj_prp_rel_df=obj_prp_rel_df
                                , att_form_def_df=att_form_def_df
                                )

Connection to Strategy One Intelligence Server has been established.
Project selected in Connection object:
Project object named: 'MicroStrategy Tutorial' with ID: 'B7CA92F04B9FAE8D941C3E9B7E0CD754'
_Cube object named: 'attribute_form_elements' with ID: '7065AA4C48A2044AF0BEED95042A4870'


Initializing an instance of a cube. Please wait...

Downloading: 100%|##########| 2/2 [00:01<00:00,  1.01it/s, rows=30217]

_Cube object named: 'attribute_elements' with ID: 'E098E8DD49B73D9FBE4F17ACC6774F3E'


Initializing an instance of a cube. Please wait...

Downloading: 100%|##########| 2/2 [00:00<00:00, 13.07it/s, rows=1029]

_Cube object named: 'att_form_def' with ID: 'A8B8EF0949433604826137AC569859E3'


Initializing an instance of a cube. Please wait...

_Cube object named: 'obj_prp_rel' with ID: '21BCCD6C4110696EABE622B71D1C0566'


Initializing an instance of a cube. Please wait...

### Run report

In [4]:
# tool agnostic definitions. Only MSTR for the moment
element_df_d_l=[{"df":attribute_form_elements_df,"key_col":"element_val","key_type":"element_val","rag_cols": ["attribute_name", "form_name", "element_val"]},
                 {"df":attribute_elements_df,"key_col":"element_val","key_type":"element_val","rag_cols": ["attribute_name", "element_val"]}
                ]

bi_obj_df=obj_prp_rel_df[["object_name", "obj_type", "object_id"]][obj_prp_rel_df["obj_type"].isin(["attribute","metric"]) ]

obj_df_d_l=[{"df":bi_obj_df,"key_col":"object_name","key_type":"object_name","rag_cols": ["object_name", "obj_type"]}]
            


In [5]:
key_word_l=u_keyword_processor.extract_keywords(msg_t=msg_t)

att_elem_str=i_mstr_objects.get_att_elem_str(element_df_d_l, key_word_l=key_word_l)
bi_obj_str=i_mstr_objects.get_att_elem_str(obj_df_d_l, key_word_l=key_word_l)

sys_cont=u_perplexity.rag_sys_cont(key_word_l=key_word_l,att_elem_str=att_elem_str, bi_obj_str=bi_obj_str)

message_check_d={}
message_check_d_l=[]
message_check_d["msg_nr"] = "1"
message_check_d["msg_t"] = msg_t
message_check_d=u_perplexity.call_perplexity( msg_t=msg_t, sys_cont=sys_cont, message_check_d=message_check_d, temperature=0.1)
message_check_d_l.append(message_check_d.copy())

bi_request_d=u_perplexity.parse_and_structure(message_check_d_l)
bi_request_d

{'msg_nr': '1',
 'msg_t': 'Please show me the Cost_1, Revenue and Profit for the attributes Year,Category, Region and filter for the yaer 2021, 2022 and 2023 and Categories starting with B and Revenue is between 10 and 1000000',
 'attributes': ['Year', 'Category', 'Region'],
 'metrics': ['Cost', 'Revenue', 'Profit'],
 'filter': {'att_element_1': {'attribute': 'Year',
   'operator': 'In',
   'element_list': ['2021', '2022', '2023']},
  'att_qual_1': {'attribute': 'Category',
   'Column': 'desc',
   'operator': 'BeginsWith',
   'value': 'B'},
  'metric_filter_1': {'level': None,
   'metric': 'Revenue',
   'operator': 'Between',
   'value': [10, 1000000]}},
 'question': ''}

### export data

In [ ]:
prompt_answ=i_answer_prompts.AI_mstr_prp_page_ans( conn=conn
                                                  ,vector_store=u_keyword_processor
                                                  ,bi_request_d=bi_request_d
                                                  ,rep_dos_id=report_id
                                                  )

rep_id=i_answer_prompts.save_AI_rep(conn=conn,report_id=report_id
                                  ,prompt_answ=prompt_answ
                                  ,ai_rep_name=ai_rep_name
                                  ,ai_rep_folder_id=ai_rep_folder_id)
new_rep_id=rep_id.json()["id"]
instance_id = i_rep.open_Instance(conn=conn, report_id=new_rep_id)
df=i_rep.report_df(conn=conn, report_id=new_rep_id, instance_id=instance_id)
df

Report object named: 'dyn_prompt_page_botstat' with ID: 'C731B12148180FDA527FB5BCE95E618E'


Initializing an instance of a report. Please wait...

,Category,Region,Year,Profit,Revenue,Cost
0,Books,Central,2021,26768.360,124045.80,97277.440
1,Books,Central,2022,33372.717,154588.50,121215.783
2,Books,Mid-Atlantic,2021,24850.750,114815.95,89965.200
3,Books,Mid-Atlantic,2022,29539.700,136809.85,107270.150
4,Books,Northeast,2021,47111.068,218225.70,171114.632
5,Books,Northeast,2022,57385.104,263193.85,205808.746
6,Books,Northwest,2021,9961.775,45521.65,35559.875
7,Books,Northwest,2022,10767.738,49594.50,38826.762
8,Books,South,2021,28260.394,133762.05,105501.656
9,Books,South,2022,35261.976,164573.60,129311.624


### verify report in MSTR

In [8]:
rep_id=i_answer_prompts.save_AI_rep(conn=conn,report_id=report_id
                                  ,prompt_answ=prompt_answ
                                  ,ai_rep_name=ai_rep_name
                                  ,promptOption ="filterAndTemplate"
                                  ,ai_rep_folder_id=ai_rep_folder_id)
new_rep_id=rep_id.json()["id"]
link=i_rep.web_base_url(conn=conn,report_id=new_rep_id)
HTML(link)

http://217.154.213.84:8080/MicroStrategy/servlet/mstrWeb?Server=217.154.213.84&Project=MicroStrategy+Tutorial&evt=4001&src=mstrWeb.4001&reportViewMode=1&reportID=C731B12148180FDA527FB5BCE95E618E&currentViewMedia=2
